# Day 18 · 第一次训练收官

**配套讲义**: [`days/day-18.md`](../days/day-18.md) ｜ **需要 GPU（云机器）**

合并 LoRA → 导出可独立加载的权重，然后做 **20 条 base vs 你的版本的人工并排抽检** —— 第一次能回答「我训的模型到底有没有变好」。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w3.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 合并 LoRA 并导出

In [ ]:
import sys; sys.path.insert(0, "..")
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from peft import PeftModel

BASE = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER = "outputs/qwen25vl3b-cx-lora-v0"
OUT = "outputs/qwen25vl3b-cx-merged-v0"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE, torch_dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()          # ★ 关键一步
model.save_pretrained(OUT)
AutoProcessor.from_pretrained(BASE).save_pretrained(OUT)
print("已导出 →", OUT)

## 2. 生成 20 条并排对比

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.eval.quick_eval",
                    "--base", "Qwen/Qwen2.5-VL-3B-Instruct",
                    "--adapter", "outputs/qwen25vl3b-cx-lora-v0",
                    "--n", "20", "--out", "reports/quick_eval_w3.md"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-2000:] or r.stderr[-2000:])

## 3. 逐条打分表格（今天的核心产出）

把这 20 条复制到下面，逐条写判断。**建议直接看 `reports/quick_eval_w3.md` 的表格**，
在这里只填你的结论汇总。

In [ ]:
verdict = {
    "变好": [],   # 例: ["L1-有货吗-答出了具体码数", ...]
    "变差": [],
    "没变": [],
}
for k, v in verdict.items():
    print(f"{k}: {len(v)} 条")

hypothesis = """
变好的共性：
变差的共性：
下一轮要补的数据：
"""
print(hypothesis)

## 验收清单

- [ ] 合并后的权重能用**一个独立的 `from_pretrained` 直接加载**（不挂 adapter）
- [ ] 20 条并排对比已生成，且**你逐条写了判断**
- [ ] 能指出至少 **3 个变好的 case** 和 **2 个变差的 case**，并给出原因假设
- [ ] `progress/weekly-review.md` 的 W3 段已写；进度表 W3 六天 `[x]`，M3 打卡

**卡住了？** 回看 [`days/day-18.md`](../days/day-18.md) 第五节「容易踩的坑」。

> **明天**：`days/day-19.md` —— W4 评测周，今天起可以回本地做（省钱）